In [0]:
%pip install faker

In [0]:
from faker import Faker
import random
import pandas as pd

# Seed everything for reproducibility -- critical for a research paper,
# so results can be independently regenerated and verified.
SEED = 42
random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

In [0]:
def generate_base_person(person_id: int):
    gender = random.choice(["M", "F"])
    first_name = fake.first_name_male() if gender == "M" else fake.first_name_female()
    last_name = fake.last_name()
    dob = fake.date_of_birth(minimum_age=0, maximum_age=95)

    address_1 = fake.street_address()
    # ~35% of people have a unit/apt/suite -- realistic mix of house vs apartment dwellers
    has_address_2 = random.random() < 0.35
    address_2 = fake.secondary_address() if has_address_2 else None

    city = fake.city()
    state = fake.state_abbr()
    zip_code = fake.zipcode()

    ssn = fake.ssn()
    phone = fake.phone_number()

    return {
        "master_person_id": person_id,   # GROUND TRUTH KEY ONLY -- never used as a match feature
        "first_name": first_name,
        "last_name": last_name,
        "gender": gender,
        "dob": dob.isoformat(),
        "address_1": address_1,
        "address_2": address_2,          # can be None
        "city": city,
        "state": state,
        "zip": zip_code,
        "ssn": ssn,                      # true value; per-payer presence decided in Step 2
        "phone": phone,                  # true value; per-payer presence decided in Step 2
    }

N_PEOPLE = 1000  # start small -- we'll scale up later once the pipeline is validated

base_population = [generate_base_person(i) for i in range(1, N_PEOPLE + 1)]

# Sanity check before moving to Spark
pd.DataFrame(base_population).head(10)

In [0]:
base_df = spark.createDataFrame(base_population)
base_df.printSchema()
base_df.show(5, truncate=False)

In [0]:
# Save as a Delta table -- this is our permanent, untouched "answer key".
# We will NEVER modify this table directly; every payer file in later
# steps will be a noisy, partial DERIVED COPY of rows from this table.

base_df.write.format("delta").mode("overwrite").saveAsTable("bronze.payer.synthetic_base_population")

print(f"Ground truth population saved: {base_df.count()} people")

In [0]:
# Each payer has its own tendency to populate optional fields.
# This mirrors real-world inconsistency: legacy payers often still
# capture SSN; newer/post-MBI-transition payers often don't.

payer_policies = {
    "PAYER_A": {"ssn_fill_rate": 0.90, "phone_fill_rate": 0.85, "addr2_fill_rate": 0.95},
    "PAYER_B": {"ssn_fill_rate": 0.10, "phone_fill_rate": 0.60, "addr2_fill_rate": 0.80},
    "PAYER_C": {"ssn_fill_rate": 0.50, "phone_fill_rate": 0.95, "addr2_fill_rate": 0.70},
    "PAYER_D": {"ssn_fill_rate": 0.05, "phone_fill_rate": 0.40, "addr2_fill_rate": 0.90},
}

payer_ids = list(payer_policies.keys())
print(payer_ids)

In [0]:
import numpy as np

np.random.seed(SEED)

# Distribution of "how many payers is this person in"
# 60% -> 1 payer, 25% -> 2 payers, 10% -> 3 payers, 5% -> all 4
overlap_distribution = {
    1: 0.60,
    2: 0.25,
    3: 0.10,
    4: 0.05,
}

def assign_payers_for_person():
    n_payers = np.random.choice(
        list(overlap_distribution.keys()),
        p=list(overlap_distribution.values())
    )
    return random.sample(payer_ids, k=n_payers)

membership_rows = []
for person in base_population:
    pid = person["master_person_id"]
    assigned_payers = assign_payers_for_person()
    for payer in assigned_payers:
        membership_rows.append({"master_person_id": pid, "payer_id": payer})

membership_df = pd.DataFrame(membership_rows)
print(f"Total membership rows: {len(membership_df)}")
print(f"Unique people: {membership_df['master_person_id'].nunique()}")
membership_df.head(10)

In [0]:
payers_per_person = membership_df.groupby("master_person_id")["payer_id"].nunique()
overlap_check = payers_per_person.value_counts(normalize=True).sort_index()
print("Actual overlap distribution achieved:")
print(overlap_check)

In [0]:
membership_spark_df = spark.createDataFrame(membership_df)
membership_spark_df.write.format("delta").mode("overwrite").saveAsTable("bronze.payer.synthetic_membership")

print("Membership table saved.")
display(membership_spark_df.limit(10))

In [0]:
import re

NOISE_RATE = 0.15  # 15% chance per field, per record

# A small nickname map -- in a real paper, you'd cite/borrow from a public
# nickname dictionary (e.g., the "Census Bureau" or open nickname datasets).
# Keeping this small and illustrative for now; can be expanded later.
NICKNAME_MAP = {
    "Robert": "Bob", "William": "Bill", "Richard": "Rick", "James": "Jim",
    "Michael": "Mike", "Elizabeth": "Liz", "Katherine": "Kate", "Margaret": "Peggy",
    "Jonathan": "Jon", "Christopher": "Chris", "Alexander": "Alex", "Nicholas": "Nick",
    "Patricia": "Pat", "Deborah": "Debbie", "Jennifer": "Jen", "Samantha": "Sam",
}

def maybe(prob=NOISE_RATE):
    return random.random() < prob

def inject_typo(name: str) -> str:
    """Swap two adjacent characters -- simulates common data-entry typos."""
    if len(name) < 3:
        return name
    i = random.randint(0, len(name) - 2)
    chars = list(name)
    chars[i], chars[i + 1] = chars[i + 1], chars[i]
    return "".join(chars)

def maybe_nickname(first_name: str) -> str:
    return NICKNAME_MAP.get(first_name, first_name)

def corrupt_name(name: str) -> str:
    if maybe():
        name = inject_typo(name)
    return name

def corrupt_dob(dob_str: str) -> str:
    """Transpose day/month digits -- a very common real-world DOB entry error."""
    if maybe():
        try:
            year, month, day = dob_str.split("-")
            # swap month/day if valid as a date (avoids creating e.g. month=13)
            if int(day) <= 12:
                return f"{year}-{day}-{month}"
        except Exception:
            pass
    return dob_str

def corrupt_address(address_1: str, address_2):
    """Simulate formatting differences: abbreviations, or merging address_2 into address_1."""
    addr1, addr2 = address_1, address_2
    if maybe():
        addr1 = (addr1.replace("Street", "St").replace("Avenue", "Ave")
                       .replace("Road", "Rd").replace("Drive", "Dr")
                       .replace("Boulevard", "Blvd"))
    if addr2 and maybe():
        # merge address_2 into address_1, and null out address_2
        # (mimics payers that use a single free-text address field)
        addr1 = f"{addr1} {addr2}"
        addr2 = None
    return addr1, addr2

def corrupt_phone(phone: str) -> str:
    """Randomize formatting: (555) 123-4567 vs 555-123-4567 vs 5551234567."""
    digits = re.sub(r"\D", "", phone)[-10:]  # keep last 10 digits
    if len(digits) < 10:
        return phone
    if maybe():
        fmt = random.choice(["dashes", "plain", "dots"])
        if fmt == "dashes":
            return f"{digits[0:3]}-{digits[3:6]}-{digits[6:]}"
        elif fmt == "dots":
            return f"{digits[0:3]}.{digits[3:6]}.{digits[6:]}"
        else:
            return digits
    return phone

def randomize_casing(name: str) -> str:
    """Independent of payer -- any record, any payer, might come through in odd casing."""
    if maybe():
        return name.upper()
    return name

In [0]:
base_lookup = {p["master_person_id"]: p for p in base_population}

def generate_payer_record(person: dict, payer_id: str, policy: dict) -> dict:
    # Start from the TRUE values
    first_name = corrupt_name(maybe_nickname(person["first_name"]))
    last_name = corrupt_name(person["last_name"])
    first_name = randomize_casing(first_name)
    last_name = randomize_casing(last_name)

    dob = corrupt_dob(person["dob"])

    address_1, address_2 = corrupt_address(person["address_1"], person["address_2"])

    # Field availability -- per payer policy (structural noise from Step 2 design)
    ssn = person["ssn"] if random.random() < policy["ssn_fill_rate"] else None
    phone_raw = person["phone"] if random.random() < policy["phone_fill_rate"] else None
    phone = corrupt_phone(phone_raw) if phone_raw else None

    # address_2 availability layered on top of whatever corrupt_address already decided
    if address_2 and random.random() >= policy["addr2_fill_rate"]:
        address_2 = None

    return {
        "payer_id": payer_id,
        "member_id": f"{payer_id}_{person['master_person_id']}_{random.randint(1000,9999)}",
        "master_person_id": person["master_person_id"],  # kept ONLY for our internal evaluation;
                                                           # would NOT exist in a real payer file
        "first_name": first_name,
        "last_name": last_name,
        "gender": person["gender"],
        "dob": dob,
        "address_1": address_1,
        "address_2": address_2,
        "city": person["city"],
        "state": person["state"],
        "zip": person["zip"],
        "ssn": ssn,
        "phone": phone,
    }

payer_records = []
for _, row in membership_df.iterrows():
    person = base_lookup[row["master_person_id"]]
    payer_id = row["payer_id"]
    policy = payer_policies[payer_id]
    payer_records.append(generate_payer_record(person, payer_id, policy))

payer_records_df = pd.DataFrame(payer_records)
print(f"Total payer records generated: {len(payer_records_df)}")
payer_records_df.head(10)

In [0]:
# Check field fill rates roughly match intended payer policies
for payer_id, policy in payer_policies.items():
    subset = payer_records_df[payer_records_df["payer_id"] == payer_id]
    ssn_rate = subset["ssn"].notna().mean()
    phone_rate = subset["phone"].notna().mean()
    print(f"{payer_id}: intended ssn={policy['ssn_fill_rate']}, actual={ssn_rate:.2f} | "
          f"intended phone={policy['phone_fill_rate']}, actual={phone_rate:.2f} | n={len(subset)}")

In [0]:
payer_records_spark_df = spark.createDataFrame(payer_records_df)

# Combined table (internal use -- includes master_person_id for evaluation)
payer_records_spark_df.write.format("delta").mode("overwrite") \
    .saveAsTable("bronze.payer.synthetic_payer_records_with_truth")

# Individual payer eligibility files -- these simulate what each payer ACTUALLY sends,
# so master_person_id must be dropped here.
for payer_id in payer_ids:
    payer_slice = payer_records_spark_df.filter(payer_records_spark_df.payer_id == payer_id) \
                                          .drop("master_person_id")
    table_name = f"bronze.payer.eligibility_{payer_id.lower()}"
    payer_slice.write.format("delta").mode("overwrite").saveAsTable(table_name)
    print(f"Saved {table_name}: {payer_slice.count()} records")